# Member 2 — CNN + Transfer Learning (VGG16)
## Cheating Detection in Exam Halls
---
**Input:** `processed_data/frames/` (من الـ Master Preprocessing)
```
Frames → tf.data → VGG16 (pretrained) → Train → Camera Test → Notification
```

## Cell 1 — Imports

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG16
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import time
import os
import warnings
warnings.filterwarnings('ignore')

tf.random.set_seed(42)
np.random.seed(42)

print('TensorFlow:', tf.__version__)
print('GPU:', len(tf.config.list_physical_devices('GPU')) > 0)

## Cell 2 — Data Loading (tf.data.Dataset)

In [ ]:
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
EPOCHS     = 100

TRAIN_DIR = 'processed_data/frames/train'
VAL_DIR   = 'processed_data/frames/val'
TEST_DIR  = 'processed_data/frames/test'
os.makedirs('saved_models', exist_ok=True)

# ── Augmentation layer (على الـ train بس) ──────────────
augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.2),
    layers.RandomTranslation(0.1, 0.1),
], name='augmentation')

# ── tf.data.Dataset ────────────────────────────────────
train_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary',
    shuffle=True,
    seed=42
)

val_ds = keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary',
    shuffle=False
)

test_ds = keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary',
    shuffle=False
)

CLASS_NAMES = train_ds.class_names
print('Classes:', CLASS_NAMES)

# ── Normalize + Augment + Prefetch (للسرعة) ────────────
def preprocess_train(images, labels):
    images = tf.cast(images, tf.float32) / 255.0
    images = augmentation(images, training=True)
    return images, labels

def preprocess_eval(images, labels):
    images = tf.cast(images, tf.float32) / 255.0
    return images, labels

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.map(preprocess_train, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds   = val_ds.map(preprocess_eval,   num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds  = test_ds.map(preprocess_eval,  num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

print('Train / Val / Test datasets ready')

## Cell 3 — Class Weights (لمعالجة الـ Imbalance)

In [ ]:
# اجمع كل الـ labels من الـ train
all_labels = np.concatenate([y.numpy() for _, y in train_ds])

weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(all_labels),
    y=all_labels.flatten()
)
class_weights = dict(enumerate(weights))

print('Class weights:', class_weights)
for i, name in enumerate(CLASS_NAMES):
    print(f'  {name}: {weights[i]:.3f}')

## Cell 4 — Build Model (VGG16)
```
VGG16 (pretrained ImageNet, frozen)
         ↓
GlobalAveragePooling2D
         ↓
Dense(256) → BatchNorm → ReLU → Dropout(0.5)
         ↓
Dense(128) → BatchNorm → ReLU → Dropout(0.3)
         ↓
Dense(1, sigmoid)
```

In [ ]:
# ── VGG16 Base (frozen) ────────────────────────────────
base = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base.trainable = False

# ── Custom Head ────────────────────────────────────────
inputs = keras.Input(shape=(224, 224, 3))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)

x = layers.Dense(256, kernel_regularizer=keras.regularizers.l2(0.001))(x)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)
x = layers.Dropout(0.5)(x)

x = layers.Dense(128, kernel_regularizer=keras.regularizers.l2(0.001))(x)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(1, activation='sigmoid')(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

## Cell 5 — Callbacks

In [ ]:
callbacks = [

    # احفظ أحسن model
    keras.callbacks.ModelCheckpoint(
        'saved_models/best_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),

    # وقف لو مفيش تحسن
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),

    # قلل الـ LR تلقائياً
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=7,
        min_lr=1e-7,
        verbose=1
    ),

    # سجل كل epoch في CSV
    keras.callbacks.CSVLogger('saved_models/training_log.csv'),

]

print('Callbacks ready')

## Cell 6 — Train

In [ ]:
history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

print('Training done!')

## Cell 7 — Plot Training History

In [ ]:
acc      = history.history['accuracy']
val_acc  = history.history['val_accuracy']
loss     = history.history['loss']
val_loss = history.history['val_loss']
epochs   = range(1, len(acc) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, acc,     label='acc')
axes[0].plot(epochs, val_acc, label='val_acc')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(epochs, loss,     label='loss')
axes[1].plot(epochs, val_loss, label='val_loss')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('saved_models/training_curves.png', dpi=150)
plt.show()

## Cell 8 — Evaluate on Test Set

In [ ]:
best_model = keras.models.load_model('saved_models/best_model.keras')

loss, acc = best_model.evaluate(test_ds, verbose=0)
print(f'Test Loss     : {loss:.4f}')
print(f'Test Accuracy : {acc:.4f}')

# Confusion Matrix
y_true, y_pred = [], []
for images, labels in test_ds:
    preds = best_model.predict(images, verbose=0)
    y_pred.extend((preds > 0.5).astype(int).flatten())
    y_true.extend(labels.numpy().flatten().astype(int))

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('saved_models/confusion_matrix.png', dpi=150)
plt.show()

## Cell 9 — Live Camera Test

بيفتح الكاميرا لمدة **30-40 ثانية**، بيعمل predict على كل frame، وفي الآخر بيبعت notification بالنتيجة.

**كاميرا الموبايل:**
1. حمّل تطبيق **IP Webcam** (Android) أو **EpocCam** (iPhone)
2. شغّل الـ app وهيديك رابط مثل: `http://192.168.1.5:8080/video`
3. حطّ الـ IP ده في `MOBILE_CAM_URL` تحت


In [5]:
import threading

# ── إعدادات الكاميرا ───────────────────────────────────
USE_LAPTOP_CAM  = True          # True = كاميرا اللاب
                                # False = كاميرا الموبايل
MOBILE_CAM_URL  = 'http://192.168.1.5:8080/video'  # ← غيّر الـ IP بتاعك
TEST_DURATION   = 35            # ثواني (30-40)
CHEAT_THRESHOLD = 0.5           # نسبة الـ cheating frames عشان يعتبره غش

# ── Notification بصوت + رسالة ──────────────────────────
def send_notification(result, cheat_ratio):
    msg = ''
    if result == 'CHEATING':
        msg = f'⚠️  تحذير: تم اكتشاف غش!  ({cheat_ratio:.0%} من الوقت)'
    else:
        msg = f'✅  لا يوجد غش  ({1 - cheat_ratio:.0%} طبيعي)'

    # رسالة على الشاشة
    print('\n' + '='*50)
    print(msg)
    print('='*50)

    # صوت تنبيه
    try:
        import winsound   # Windows
        if result == 'CHEATING':
            for _ in range(3):
                winsound.Beep(1000, 400)
                time.sleep(0.1)
        else:
            winsound.Beep(600, 500)
    except ImportError:
        try:
            import subprocess   # Mac / Linux
            if result == 'CHEATING':
                subprocess.run(['say', 'Warning cheating detected'], capture_output=True)
            else:
                subprocess.run(['say', 'No cheating detected'], capture_output=True)
        except:
            print('(تشغيل الصوت مش متاح على هذا النظام)')

# ── Predict على frame واحدة ────────────────────────────
def predict_frame(frame, loaded_model):
    img = cv2.resize(frame, IMG_SIZE)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img.astype(np.float32) / 255.0
    img = np.expand_dims(img, axis=0)
    prob = loaded_model.predict(img, verbose=0)[0][0]
    # CLASS_NAMES مرتبة أبجدياً: ['cheating'=0, 'normal'=1]
    label = CLASS_NAMES[int(prob > 0.5)]
    return label, prob

# ── Main Camera Loop ───────────────────────────────────
def run_camera_test():
    loaded_model = keras.models.load_model('abanoub_cheating_detection_results/best_model.keras')

    # اختار مصدر الكاميرا
    if USE_LAPTOP_CAM:
        cap = cv2.VideoCapture(0)
        print('كاميرا اللاب شغالة...')
    else:
        cap = cv2.VideoCapture(MOBILE_CAM_URL)
        print(f'كاميرا الموبايل: {MOBILE_CAM_URL}')

    if not cap.isOpened():
        print('ERROR: مش قادر يفتح الكاميرا!')
        return

    print(f'\nالتسجيل بدأ — هينتهي بعد {TEST_DURATION} ثانية')
    print('اضغط Q للإيقاف المبكر')

    start_time    = time.time()
    total_frames  = 0
    cheat_frames  = 0
    predict_every = 10   # predict كل 10 frames عشان السرعة
    frame_count   = 0
    last_label    = 'normal'
    last_prob     = 0.0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        elapsed = time.time() - start_time
        remaining = TEST_DURATION - elapsed

        # Predict كل predict_every frames
        if frame_count % predict_every == 0:
            last_label, last_prob = predict_frame(frame, loaded_model)
            total_frames += 1
            if last_label == 'cheating':
                cheat_frames += 1

        # ── ارسم على الـ frame ──────────────────────────
        color = (0, 0, 255) if last_label == 'cheating' else (0, 200, 0)
        text  = f'{last_label.upper()}  ({last_prob:.0%})'
        cv2.rectangle(frame, (0, 0), (frame.shape[1], 50), (0, 0, 0), -1)
        cv2.putText(frame, text, (10, 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2)
        cv2.putText(frame, f'Time left: {remaining:.0f}s', (10, frame.shape[0] - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)

        cv2.imshow('Cheating Detection Test', frame)

        frame_count += 1

        # وقف لما الوقت خلص
        if elapsed >= TEST_DURATION:
            break

        # وقف لما اضغط Q
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print('تم الإيقاف يدوياً')
            break

    cap.release()
    cv2.destroyAllWindows()

    # ── النتيجة النهائية ────────────────────────────────
    if total_frames == 0:
        print('لم يتم التقاط أي frames!')
        return

    cheat_ratio = cheat_frames / total_frames
    final_result = 'CHEATING' if cheat_ratio >= CHEAT_THRESHOLD else 'NORMAL'

    print(f'\nFrames analyzed : {total_frames}')
    print(f'Cheating frames : {cheat_frames} ({cheat_ratio:.0%})')

    send_notification(final_result, cheat_ratio)


# ── شغّل الـ test ──────────────────────────────────────
run_camera_test()

NameError: name 'keras' is not defined